In [1]:
"""import warnings
# Глушым стандартны клас папярэджанняў Python
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Глушым любыя папярэджанні, якія ўтрымліваюць тэкст пра eval_set
warnings.filterwarnings("ignore", message=".*eval_set.*is deprecated.*")
# Глушым менавіта гэтае папярэджанне па яго тэксту
warnings.filterwarnings("ignore", message=".*The argument 'eval_set' is deprecated.*")"""

'import warnings\n# Глушым стандартны клас папярэджанняў Python\nwarnings.filterwarnings("ignore", category=DeprecationWarning)\n\n# Глушым любыя папярэджанні, якія ўтрымліваюць тэкст пра eval_set\nwarnings.filterwarnings("ignore", message=".*eval_set.*is deprecated.*")\n# Глушым менавіта гэтае папярэджанне па яго тэксту\nwarnings.filterwarnings("ignore", message=".*The argument \'eval_set\' is deprecated.*")'

In [2]:
import sys
from pathlib import Path

from modules.dataset_regression import PrepareRegressionDataset
from modules.regressor import RegressorBench
from modules.regression_evaluation import RegressionEvaluator
from modules.regression_tuner import BayesianRegressionTuner
from modules.visualization import DataVisualizer

ROOT = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

DATA = ROOT / "data"
OUT = ROOT / "output"

d:\DScourse\hw5\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
OUT.mkdir(parents=True, exist_ok=True)
dataset = PrepareRegressionDataset(DATA)
bench = RegressorBench()
evaluator = RegressionEvaluator()
viz = DataVisualizer(OUT)

#section("1. Датасет: рэгрэсія PageValues")
frame = dataset.load_csv(file_name="online_shoppers_intention.csv")

ohe_feature_list = ["OperatingSystems", "Browser", "Region", "TrafficType", "VisitorType"]
known_categories = {
    col: sorted(list(frame[col].dropna().unique())) for col in ohe_feature_list
}

# Колькасная дыягностыка таргету і прыкмет
dataset.generate_diagnostic_table(frame, target_name="PageValues")  

[LoadDataset] online_shoppers_intention.csv: 12330 радкоў x 18 слупкоў
| Уласцівасць | Значэнне |
| :--- | :--- | :--- |
| Доля нулявых PageValues | 77.86 % |
| Skewness / Kurtosis | 6.38 / 65.6 |
| Max корреляцыя з прыкметамі | 0.17 (ExitRates) |
| BounceRates ↔ ExitRates | r = 0.91 |
| ProductRelated ↔ ProductRelated_Duration | r = 0.86 |
| Кардынальнасць TrafficType / Browser | 20 / 13 катэгорый |


### Колькасная дыягностыка таргету і прыкмет
| Уласцівасць | Значэнне | Наступства для выбару мадэлі |
| :--- | :--- | :--- |
| **Доля нулявых PageValues** | 77.86 % | Задача з залішкам нулёў (zero-inflated); патрэбны мадэлі, здольныя ставіць дакладны 0 |
| **Асіметрыя / Эксцэс** | 6.38 / 65.6 | Экстрэмальна цяжкі хвост; лінейныя мадэлі і MSE-стратэгіі адчувальныя да выкідаў |
| **Max каррэляцыя з прыкметамі** | 0.17 (ExitRates) | Сувязь нелінейная; лінейныя мадэлі структурна не падыходзяць |
| **BounceRates ↔ ExitRates** | r = 0.91 | Моцная мультыкалінеарнасць; патрабуе рэгулярызацыі (Ridge) або дрэвападобных мадэляў |
| **ProductRelated ↔ ProductRelated_Duration** | r = 0.86 | Моцная мультыкалінеарнасць; патрабуе рэгулярызацыі (Ridge) або дрэвападобных мадэляў |
| **Кардынальнасць TrafficType / Browser** | 20 / 13 катэгорый | OHE моцна павялічвае прастору прыкмет; CatBoost/дрэвы пераносяць гэта лепш за KNN ці лінейныя мадэлі |


Гэтая дыягностыка непасрэдна вызначыла выбар з 19 прапанаваных метадаў рэгрэсіі:   
 * пераважная стаўка зроблена на нелінейныя ансамблі (LightGBM, XGBoost, CatBoost, ExtraTrees, DecisionTree),   
 * лінейная мадэль пакінута толькі адна (Ridge) як кантрольны прыклад таго, чаму лінейнасць тут не працуе,   
 * KNN, AdaBoost, OMP, Passive-Aggressive і інш. былі адхілены як структурна недарэчныя.

### 2. Падзел 60/20/20 (Revenue выдалены з прыкмет, стратыфікацыя па бінах PageValues)

In [4]:
split = dataset.prepare_regression(
    frame,
    target_name="PageValues",
    test_size=0.2,
    val_size=0.2,
    random_state=42,
)

bench.build()
bench.preprocessor = dataset.create_preprocessor_regression(categories_dict=known_categories)

# section("2.1. Дадаем 8-ы метад: TwoStageRegressor (hurdle-мадэль)")
    # Асобны шлях: класіфікатар-варотнік P(PageValues>0) + LightGBM-рэгрэсія
    # combine="gated" (дэфолт).
bench.add_two_stage_model(name="TwoStage")

[PrepareRegressionDataset] Радкоў: 12330, таргет = PageValues
  Нулявых значэнняў: 77.86%, сярэдняе=5.889, медыяна=0.000, max=361.764
[PrepareRegressionDataset] Выдалены з прыкмет (абарона ад уцёку): ['Revenue']
[PrepareRegressionDataset] Падзел static_3way (60/20/20): train=7398, val=2466, test=2466
[PrepareRegressionDataset] Падзел пад CV + hold-out: CV=9864, test=2466
[RegressorBench] Метады:
  Dummy Regressor      — baseline (медыяна, улічвае 78% нулёў)
  Ridge                — лінейная, L2 супраць мультыкалінеарнасці
  DecisionTree         — простае нелінейнае дрэва
  ExtraTrees           — бэггінг
  LightGBM, XGBoost, CatBoost — бустынг
[RegressorBench] Дададзены 8-ы метад: TwoStage (hurdle-мадэль, combine='gated', threshold=0.5)


### 3. Крос-валідацыя (5 фолдаў, стратыфікаваных па нуль/квантылі PageValues)

In [5]:
CV_methods = ["Dummy", "Ridge", "DecisionTree", "ExtraTrees", "LightGBM", "XGBoost", "CatBoost", "TwoStage"]
CV_methods = [m for m in CV_methods if m in bench.models]  # прапусціць CatBoost, калі не ўстаноўлены
CV_configs = bench.prepare_final_models(models_to_keep=CV_methods)

fold_rmse, fold_mae, fold_r2, oof_predictions = bench.cross_validate_newV(
    split.x_cv, split.y_cv, models_conf=CV_configs
)

metrics_table = bench.get_metrics_table()
print("\n[Cross-Validation] Табліца метрык, адсартавана па RMSE:")
print(metrics_table.to_string(index=False))

df_report = evaluator.compile_performance_report(y_true=split.y_cv, preds_dict=oof_predictions)
print("\n[Report] Поўная справаздача OOF:")
print(df_report.to_string(index=False))

[Cross-Validation] Dummy ...
     RMSE=19.6571 ± 1.8986   MAE=5.9002 ± 0.2081   R2=-0.1007 ± 0.0123
[Cross-Validation] Ridge ...
     RMSE=18.2552 ± 1.8930   MAE=9.1864 ± 0.2107   R2=0.0519 ± 0.0120
[Cross-Validation] DecisionTree ...
     RMSE=18.0268 ± 2.0312   MAE=8.1730 ± 0.1637   R2=0.0770 ± 0.0189
[Cross-Validation] ExtraTrees ...
     RMSE=18.0353 ± 1.8977   MAE=8.2316 ± 0.1563   R2=0.0747 ± 0.0176
[Cross-Validation] LightGBM ...
     RMSE=17.6935 ± 1.9556   MAE=8.1438 ± 0.1958   R2=0.1103 ± 0.0235
[Cross-Validation] XGBoost ...
     RMSE=17.6998 ± 2.0068   MAE=8.1671 ± 0.1823   R2=0.1101 ± 0.0254
[Cross-Validation] CatBoost ...
     RMSE=17.6736 ± 1.9996   MAE=8.1672 ± 0.1873   R2=0.1127 ± 0.0224
[Cross-Validation] TwoStage ...
     RMSE=19.5976 ± 1.7319   MAE=6.4513 ± 0.1489   R2=-0.0961 ± 0.0341

[Cross-Validation] Табліца метрык, адсартавана па RMSE:
       model RMSE (Mean ± STD) MAE (Mean ± STD)  R2 (Mean ± STD)
    CatBoost  17.6736 ± 1.9996  8.1672 ± 0.1873  0.1127 ± 0.0

In [6]:
viz.plot_regression_metric_comparison(
    fold_rmse, metric_name="RMSE", lower_is_better=True, filename="regression_rmse_comparison.png",
)
viz.plot_regression_metric_comparison(
    fold_mae, metric_name="MAE", lower_is_better=True, filename="regression_mae_comparison.png",
)
viz.plot_regression_metric_comparison(
    fold_r2, metric_name="R2", lower_is_better=False, filename="regression_r2_comparison.png",
)

WindowsPath('d:/DScourse/hw5/output/regression_r2_comparison.png')

![Параўнанне RMSE](output\regression_rmse_comparison.png)

![Параўнанне MAE](output\regression_mae_comparison.png)

![Параўнанне R²](output\regression_r2_comparison.png)

**Інтэрпрэтацыя вынікаў: параўнанне метадаў на крос-валідацыі**

●	Тры бустынгі (CatBoost, LightGBM, XGBoost) утвараюць яўны лідарскі кластар: RMSE 17.67–17.72, статыстычна непадзельныя паміж сабой.  
●	DecisionTree і ExtraTrees практычна ідэнтычныя (RMSE 18.03), што пацвярджае: адзінае дрэва з абмежаванай глыбінёй ужо лавіла асноўную нелінейную структуру, а бэггінг дадаў мала.  
●	Ridge (лінейная) — найгоршая сярод "сапраўдных" мадэляў: RMSE 18.26, MAE 9.19 (найгорша MAE ўвогуле сярод не-baseline мадэляў). Гэта наўпрост пацвярджае гіпотэзу з раздзела 2: слабая лінейная карэляцыя (max r = 0.17) сапраўды не дазваляе лінейнай мадэлі скарыстацца структурай даных.  
●	Dummy (медыяна = 0) мае RMSE 19.66, што горш за ўсе рэальныя мадэлі, але лепшы MAE (5.90) за ўсе, акрамя TwoStage. Гэта не супярэчнасць: пры 78% нулявых значэнняў просты прагноз "заўсёды 0" мае вельмі малую абсалютную памылку ў сярэднім, але вялікую квадратычную памылку на 22% сесій з ненулявым PageValues.  
●	R² базуецца на параўнанні з сярэднім значэннем y (не медыянай), таму нават Dummy (які прадказвае медыяну = 0) можа мець адмоўны R²: гэта артэфакт азначэння R², а не доказ дрэннай мадэлі — MAE тут больш справядлівая метрыка для Dummy.


### 4. Выбар лепшай мадэлі
**4.1 Парны t-тэст па RMSE**

In [7]:
best_name = evaluator.pick_best(fold_rmse)
print(f"\nАбраная мадэль: {best_name}")


[Evaluator] Парны t-тэст Сцюдэнта па фолдах (RMSE): CatBoost vs LightGBM
t-статыстыка=-0.321, p-value=0.7641, t_critical=2.776
[Evaluator] У распрацоўку: CatBoost. (p-value (0.7641) >= 0.05) Розніца НЕ значная паміж CatBoost і LightGBM

Абраная мадэль: CatBoost


**Высновы:** Парны t-тэст паміж CatBoost (RMSE 17.674) і LightGBM (RMSE 17.694) даў p = 0.764 — розніца незначная. Паводле прынцыпу Оккама (аднолькавая складанасць у абедзвюх, тэхнічная нічыя) абраны CatBoost. LightGBM з'яўляецца раўнацэннай альтэрнатывай — розніца ў RMSE (0.02) на два парадкі меншая за std (2.0).

4.2. Дыягностыка нулявых прагнозаў (zero-inflation)

In [8]:
diag_models = [best_name] + (["TwoStage"] if "TwoStage" in oof_predictions and best_name != "TwoStage" else [])
viz.plot_zero_prediction_diagnostic(
    y_true=split.y_cv.to_numpy(), oof_predictions=oof_predictions, model_names=diag_models,
    filename="regression_zero_prediction_diagnostic.png",
)

WindowsPath('d:/DScourse/hw5/output/regression_zero_prediction_diagnostic.png')

### 5. TwoStageRegressor: бар'ерная мадэль (hurdle-мадэль) для zero-inflated таргету

TwoStageRegressor — асобны sklearn-сумяшчальны wrapper (fit/predict), пабудаваны незалежна ад RegressorBench:  
●	Stage 1 (Gate): класіфікатар прадказвае P(PageValues > 0 | X)  
●	Stage 2 (Regressor): рэгрэсар навучаны толькі на радках, дзе PageValues > 0 (22.14% ад усіх сесій)  
●	Combine: фінальны прагноз камбінуе абедзве стадыі — рэжым "probability" (множанне на імавернасць) альбо "gated" (цвёрды парог)  
5.1. **Рэжым "probability"**: 
Першая канфігурацыя (class_weight="balanced" у класіфікатары + combine="probability") дала RMSE = 19.27, MAE = 10.70, R² = −0.061 — горш за ўсе астатнія мадэлі, уключаючы Dummy. Дыягностыка на асобнай валідацыйнай выбарцы паказала прычыну:
| Канфігурацыя	| RMSE	| MAE	| False-positive rate на сапраўды нулявых сесіях |
| ------- | -------- | ------ | ------- | 
| balanced + probability |	21.14	|	11.10	| 99.8 % | 
| unbalanced + probability	|	19.88	|	8.30	|	99.8 % | 
| balanced + gated@0.5 | 24.12	| 10.95	| 27.4 % | 
| unbalanced + gated@0.5 | 21.51	| 6.70	| 6.7 % | 
| unbalanced + gated@0.3 | 23.43	| 9.61	| 21.3 % | 

**Прычына правалу рэжыму "probability"**: P(nonzero) практычна ніколі не роўна дакладна нулю, таму множанне імавернасці на прагноз рэгрэсара амаль заўсёды дае малы, але ненулявы прагноз. У выніку 99.8% сапраўды нулявых сесій атрымліваюць памылковы ненулявы прагноз — і гэтыя малыя памылкі, памножаныя на 78% датасэта, руйнуюць MAE. Балансаванне класаў (class_weight="balanced") дадаткова завышае P(nonzero), пагаршаючы сітуацыю.  

5.2. **Рэжым gated**
На аснове дыягностыкі, для поўнай 5-фолдавай крос-валідацыі абрана канфігурацыя unbalanced + gated. Вынік: RMSE = 19.77 ± 1.67, MAE = 6.59 ± 0.10, R² = −0.117 ± 0.045.

![Размеркаванне прагнозаў сапраўдных нулявых сесій](output\regression_zero_prediction_diagnostic.png)

На графику LightGBM (адна стадыя, злева) сістэматычна прадказвае невялікія станоўчыя значэнні (false-positive rate 82.5%); TwoStage (gated, справа) карэктна прадказвае амаль дакладны 0 (false-positive rate 6.5%).  

**Высновы:** нетрывіяльны, але зразумелы кампраміс:  
●	TwoStage (gated) мае найлепшы MAE сярод усіх 8 мадэляў (6.59, лепш за LightGBM 8.14 і нават за Dummy 5.90 амаль параўнальна) — ён значна лепш вызначае, якія сесіі маюць PageValues = 0.
Але TwoStage мае найгоршы R² і адзін з найгорших RMSE — ён горш захоплівае веліч самога ненулявога хваста, бо рэгрэсар другой стадыі навучаны толькі на 22% даных і губляе агульны кантэкст размеркавання.  
●	Практычны вывад: выбар паміж адна-стадыйнай мадэллю (CatBoost/LightGBM) і бар’ернай мадэллю (TwoStage) залежыць ад БІЗНЕС-МЭТЫ. Калі важна дакладна пазначыць "гэтая сесія амаль дакладна не мела каштоўнасці" — TwoStage лепш. Калі важна ацаніць велічыню PageValues для ранжыравання сесій (напрыклад, для мадэлі канверсіі) — CatBoost/LightGBM лепш.


### 5. Баесаўская аптымізацыя гіперпараметраў (Optuna)

In [9]:
tuner = BayesianRegressionTuner(bench_instance=bench, random_state=42)
optimized_results = tuner.tune_or_load(best_name, split.x_cv, split.y_cv)

print("СПРАВАЗДАЧА ПАДБОРУ ГІПЕРПАРАМЕТРАЎ:")
for m_name, params in optimized_results.items():
    print(f"Мадэль: {m_name}")
    for p_key, p_val in params.items():
        print(f"  -> {p_key}: {p_val:.5f}" if isinstance(p_val, float) else f"  -> {p_key}: {p_val}")


[Tuner Cache] Знойдзены захаваны файл: output\optuna_best_params_regression.json
СПРАВАЗДАЧА ПАДБОРУ ГІПЕРПАРАМЕТРАЎ:
Мадэль: CatBoost
  -> learning_rate: 0.07951
  -> depth: 6
  -> l2_leaf_reg: 2.41318
  -> bagging_temperature: 0.74258


### 6. Фінальны рэфіт і тэст

In [10]:
final_configs = bench.prepare_final_models(
    best_name=best_name,
    optimized_results=optimized_results,
    models_to_keep=[best_name],
)
final_pipeline = bench.final_secure_refit(
    final_configs=final_configs,
    x_train=split.x_train, y_train=split.y_train,
    x_val=split.x_val, y_val=split.y_val,
)

test_preds = final_pipeline.predict(split.x_test)
test_report = evaluator.compile_performance_report(
    y_true=split.y_test,
    preds_dict={f"{best_name} (Фінальны тэст)": test_preds},
)
print("\n[Report] Метрыкі на тэставай выбарцы:")
print(test_report.to_string(index=False))


[Report] Метрыкі на тэставай выбарцы:
                  Мадэль      RMSE      MAE    MedAE       R2
CatBoost (Фінальны тэст) 16.656867 8.023945 4.109207 0.084266


**Высновы:** CatBoost навучаны на поўнай train-выбарцы (Early Stopping па val), затым ацэнены на HOLD-OUT тэставай выбарцы (2 466 сесій, якія не ўдзельнічалі ні ў CV, ні ў падборы гіперпараметраў) даў вынік `RMSE` (16.65) нават крыху лепшы за сярэдні CV `RMSE` (17.67 ± 2.00) — розніца ўнутры адной std, гэта значыць мадэль не пераацэненая і паводзіны на тэсце адпавядаюць CV-чаканням.

**6.1. Візуалізацыя фінальнай мадэлі**

In [11]:
viz.plot_actual_vs_predicted(
    y_true=split.y_test.to_numpy(), y_pred=test_preds, model_name=best_name,
    filename=f"regression_actual_vs_predicted_{best_name}.png",
)

WindowsPath('d:/DScourse/hw5/output/regression_actual_vs_predicted_CatBoost.png')

![Сапраўдны vs прадказаны PageValues на тэставай выбарцы (CatBoost)](output/regression_actual_vs_predicted_CatBoost.png)

Відавочна, мадэль карэктна групуе большасць нулявых/малых сесій каля дыяганалі, але сістэматычна недаацэньвае экстрэмальныя значэнні (>100) — чакана, бо такіх сесій вельмі мала ва ўсіх выбарках.

In [12]:
viz.plot_regression_feature_importance(
    model_pipeline=final_pipeline, model_name=best_name,
    filename=f"regression_feature_importance_{best_name}.png",
)

WindowsPath('d:/DScourse/hw5/output/regression_feature_importance_CatBoost.png')

![Важнасць прыкмет](output/regression_feature_importance_CatBoost.png)

`ExitRates` — найважнейшая прыкмета (31.1%), што адпавядае найвышэйшай (па модулю) карэляцыі з `PageValues`, знойдзенай яшчэ на этапе дыягностыкі (r = −0.174). `ProductRelated` і `ProductRelated_Duration` разам складаюць яшчэ ~24.5% важнасці, нягледзячы на іх узаемную карэляцыю (r = 0.86) — дрэвападобная мадэль спраўляецца з мультыкалінеарнасцю значна лепш, чым паказаў бы каэфіцыент `Ridge`.

## 7. Высновы і стратэгія развіцця

●	Пацверджана папярэдняе абгрунтаванне выбару метадаў: усе тры бустынгі (CatBoost, LightGBM, XGBoost) статыстычна раўназначныя лідары; лінейная мадэль (Ridge) заканамерна найгоршая сярод рэальных мадэляў з-за нелінейнасці і мультыкалінеарнасці.  
●	Абсалютная якасць прагнозу (`R²` ≈ 0.09–0.11) сціплая: гэта чакана, бо ніводная асобная прыкмета не мае карэляцыі з PageValues мацней за 0.17. PageValues, відаць, нясе інфармацыю (напрыклад, пра канкрэтныя наведаныя старонкі), якая НЕ закадзіравана ў гэтых session-ўзроўневых агрэгатах.  
●	TwoStageRegressor — не "заўсёды лепшая" альтэрнатыва, а іншы аптымізацыйны кампраміс: найлепшы `MAE` цаной найгоршага `R²/RMSE`. Гэта карысны інструмент, калі бізнес-задача акцэнтуе класіфікацыю "нуль/ненуль", а не дакладную велічыню.  
●	Дадатковая аптымізацыя гіперпараметраў (BayesianRegressionTuner, Optuna) можа крыху палепшыць CatBoost/LightGBM/XGBoost, але з улікам таго, што розніца паміж усімі трыма бустынгамі і так статыстычна незначная, а столь якасці, хутчэй за ўсё, вызначаецца недахопам моцна прадказвальных прыкмет, а не недастатковым падборам гіперпараметраў.  
●	Магчымыя далейшыя крокі: log1p-трансфармацыя таргету перад навучаннем лінейных мадэляў; квантыльная рэгрэсія для лепшага ахопу хваста; патрбны больш дэталёвыя прыкметы (напрыклад,  канкрэтныя тыпы старонак).  
